# Práctica 2. Herramientas de análisis de imagen de IA: YOLO

## 1. YOLO en imagen

#### Generación del modelo YOLO

In [28]:
from ultralytics import YOLO

model_yolo = YOLO("yolo26n.pt")

#### Detección en una imagen

In [30]:
# Análisis en una imagen

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

results = model_yolo("images/perros.jpg")

results[0].show()


image 1/1 c:\Users\34634\PAI_practicas_2_3\images\perros.jpg: 448x640 2 dogs, 84.5ms
Speed: 2.5ms preprocess, 84.5ms inference, 0.7ms postprocess per image at shape (1, 3, 448, 640)


In [3]:
# Análisis de la estructura de datos de los resultados

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

result = results[0]

boxes = result.boxes

for box, cls, conf in zip(
    boxes.xyxy,
    boxes.cls,
    boxes.conf
):
    class_id = int(cls)
    class_name = model_yolo.names[class_id]
    
    print(
        f"Clase: {class_id} | "
        f"Nombre: {class_name:15s} | "
        f"Confianza: {float(conf):.3f} | "
        f"Box: {box.tolist()}"
    )

Clase: 5 | Nombre: bus             | Confianza: 0.924 | Box: [0.0, 229.36895751953125, 806.0961303710938, 757.0859375]
Clase: 0 | Nombre: person          | Confianza: 0.913 | Box: [222.469482421875, 404.9410400390625, 345.4056701660156, 861.6446533203125]
Clase: 0 | Nombre: person          | Confianza: 0.905 | Box: [47.68769073486328, 399.3199157714844, 239.34556579589844, 903.0217895507812]
Clase: 0 | Nombre: person          | Confianza: 0.870 | Box: [670.6757202148438, 391.5275573730469, 809.8081665039062, 878.9835815429688]
Clase: 0 | Nombre: person          | Confianza: 0.534 | Box: [0.12922883033752441, 556.689208984375, 60.00409698486328, 870.0955810546875]


#### Efecto de los parámetros en el análisis

In [31]:
# Análisis del efecto de la confianza en los resultados

import time

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

confianzas = [0.25, 0.75]
start = []
elapsed = []
fps = []
n_objects = []

for confianza in confianzas:
    start.append(time.perf_counter())

    results = model_yolo(
    "images/bus.jpg",
    conf=confianza,
    verbose=False
)

    elapsed.append(time.perf_counter() - start[-1])
    fps.append(1 / elapsed[-1])
    results[0].show()

    n_objects.append(len(results[0].boxes))

# Imprimimos una tabla con los resultados de tiempo y FPS
print(
    f"{'Confianza':<15} {'Tiempo (ms)':<15} {'FPS':<10} {'Objetos':<10}\n"
    f"{'-'*50}\n"
)
for i, (confianza, t, f) in enumerate(zip(confianzas, elapsed, fps)):
    print(
        f"{confianza:<15} {t*1000:<15.6f} {f:<10.2f} {n_objects[i]:<10}"
    )

Confianza       Tiempo (ms)     FPS        Objetos   
--------------------------------------------------

0.25            95.884600       10.43      5         
0.75            141.451200      7.07       4         


In [32]:
# Análisis del efecto del tamaño de la imagen en los resultados

import time

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

tamanos = [320, 1280]
start = []
elapsed = []
fps = []
n_objects = []

for tamano in tamanos:
    start.append(time.perf_counter())

    results = model_yolo(
    "images/bus.jpg",
    conf=confianza,
    verbose=False
)

    elapsed.append(time.perf_counter() - start[-1])
    fps.append(1 / elapsed[-1])
    results[0].show()

    n_objects.append(len(results[0].boxes))

# Imprimimos una tabla con los resultados de tiempo y FPS
print(
    f"{'Tamaño':<15} {'Tiempo (ms)':<15} {'FPS':<10} {'Objetos':<10}\n"
    f"{'-'*50}\n"
)
for i, (tamano, t, f) in enumerate(zip(tamanos, elapsed, fps)):
    print(
        f"{tamano:<15} {t*1000:<15.6f} {f:<10.2f} {n_objects[i]:<10}"
    )

Tamaño          Tiempo (ms)     FPS        Objetos   
--------------------------------------------------

320             78.864400       12.68      4         
1280            126.939700      7.88       4         


## 2. YOLO en vídeo

#### Detección en un vídeo frame a frame

In [34]:
# Detección en un vídeo frame a frame, con visualización de resultados

import cv2

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

cap = cv2.VideoCapture("images/calle.mp4")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model_yolo(
    frame,
    verbose=False
)

    annotated = results[0].plot()

    cv2.imshow("YOLO", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

#### Detección en un vídeo completo

In [35]:
# Análisis de un vídeo

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

results = model_yolo("images/calle.mp4", save=True)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/393) c:\Users\34634\PAI_practicas_2_3\images\calle.mp4: 384x640 3 persons, 1 truck, 72.8ms
video 1/1 (frame 2/393) c:\Users\34634\PAI_practicas_2_3\images\calle.mp4: 384x640 3 persons, 1 car, 1 truck, 70.5ms
video 1/1 (frame 3/393) c:\Users\34634\PAI_practicas_2_3\images\calle.mp4: 384x640 2 persons, 2 cars, 1 truck, 69.0ms
video 1/1 (frame 4/393) c:\Users\34634\PAI_practicas_2_3\images\calle.mp4: 384x640 1 person, 2 cars, 1 truck, 1 fire 

KeyboardInterrupt: 

## 3. Tracking YOLO

#### Tracking YOLO en un vídeo frame a frame

In [37]:
# Tracking YOLO de un vídeo frame a frame con visualización de resultados y control de reproducción

import cv2

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

video_path = "images/calle.mp4"
window_name = "YOLO Tracking"

cv2.destroyAllWindows()

cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_id = 0
playing = False

cv2.namedWindow(window_name)

try:
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()

        if not ret:
            break

        results = model_yolo.track(
    frame,
    persist=True,
    verbose=False
)

        annotated = results[0].plot()

        cv2.putText(
            annotated,
            f"Frame: {frame_id + 1}/{total_frames}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.imshow(window_name, annotated)

        key = cv2.waitKeyEx(30)
        key_low = key & 0xFF

        # Cerrar con la X
        try:
            if cv2.getWindowProperty(
                window_name,
                cv2.WND_PROP_VISIBLE
            ) < 1:
                break
        except cv2.error:
            break

        # ESC o Q
        if key == 27 or key_low == ord("q"):
            break

        # ESPACIO: reproducir/pausar
        elif key_low == ord(" "):
            playing = not playing

        # A: frame anterior
        elif key_low == ord("a"):
            playing = False
            frame_id = max(0, frame_id - 1)

        # D: frame siguiente
        elif key_low == ord("d"):
            playing = False
            frame_id = min(total_frames - 1, frame_id + 1)

        # Reproducción automática
        if playing:
            frame_id += 1

            if frame_id >= total_frames:
                frame_id = total_frames - 1
                playing = False

finally:
    cap.release()
    cv2.destroyAllWindows()

print("Reproductor de tracking cerrado.")

Reproductor de tracking cerrado.


In [38]:
# Análisis de la estructura de datos de los resultados del tracking

result = results[0]

print("Estructura de boxes:")
print(result.boxes)
print("\n-----------------------------\n")
print("IDs de los objetos detectados:")
print(result.boxes.id)
print("\n-----------------------------\n")
print("Estadísticas de las detecciones:")
print("Coordenadas de los cuadros delimitadores (xyxy):")
print(result.boxes.xyxy)
print("Confianza de las detecciones:")
print(result.boxes.conf)
print("Clases de los objetos detectados:")
print(result.boxes.cls)

Estructura de boxes:
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([10.,  2.,  2.,  0.,  7.])
conf: tensor([0.8567, 0.7671, 0.8963, 0.7264, 0.7599])
data: tensor([[1.7248e+03, 5.1249e+02, 1.9196e+03, 8.8215e+02, 4.0000e+01, 8.5668e-01, 1.0000e+01],
        [2.9722e-01, 2.8543e+02, 3.3691e+02, 6.8250e+02, 4.4000e+01, 7.6709e-01, 2.0000e+00],
        [6.9082e+02, 3.3829e+02, 1.7257e+03, 6.5140e+02, 4.8000e+01, 8.9630e-01, 2.0000e+00],
        [6.3830e+02, 3.5226e+02, 7.1377e+02, 6.0600e+02, 4.6000e+01, 7.2644e-01, 0.0000e+00],
        [6.0081e-01, 1.0809e+02, 5.1408e+02, 5.5550e+02, 5.1000e+01, 7.5988e-01, 7.0000e+00]])
id: tensor([40., 44., 48., 46., 51.])
is_track: True
orig_shape: (1080, 1920)
shape: torch.Size([5, 7])
xywh: tensor([[1822.2012,  697.3181,  194.7588,  369.6591],
        [ 168.6057,  483.9691,  336.6171,  397.0716],
        [1208.2775,  494.8464, 1034.9109,  313.1050],
        [ 676.0361,  479.1306,   75.4762,  253.7413],
        [ 257.3385,  331

#### Demostración offline

In [ ]:
# Detección YOLO en un vídeo frame a frame con tracking y guardado de trayectorias
# La visualización de los resultados se realiza en otra celda, con control de reproducción y selección de objetos

import cv2
import json
import os

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

video_path = "images/calle.mp4"
output_path = "trajectories.json"
crops_dir = "object_crops"

os.makedirs(crops_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

trajectories = {}
frame_id = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    '''
    ------------------------------------------------------------------------------------
    Escribe aquí el código para realizar la detección de objetos con tracking en
    el frame del vídeo con el modelo YOLO. El kwarg verbose debe ser False. La salida
    debe recogerse en la variable "results".
    Probar a seleccionar solamente las personas o los coches.
    ------------------------------------------------------------------------------------
    '''

    result = results[0]
    boxes = result.boxes

    if boxes.id is not None:
        for box, track_id, cls in zip(
            boxes.xyxy,
            boxes.id,
            boxes.cls
        ):
            track_id = int(track_id)
            class_id = int(cls)

            x1, y1, x2, y2 = map(int, box.tolist())

            height, width = frame.shape[:2]
            x1 = max(0, min(x1, width - 1))
            y1 = max(0, min(y1, height - 1))
            x2 = max(0, min(x2, width))
            y2 = max(0, min(y2, height))

            if x2 <= x1 or y2 <= y1:
                continue

            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            if track_id not in trajectories:
                crop_filename = f"object_{track_id}.jpg"
                crop_path = os.path.join(crops_dir, crop_filename)

                crop = frame[y1:y2, x1:x2]
                cv2.imwrite(crop_path, crop)

                trajectories[track_id] = {
                    "class_id": class_id,
                    "class_name": model_yolo.names[class_id],
                    "first_frame": frame_id,
                    "crop_path": crop_path,
                    "positions": []
                }

            trajectories[track_id]["positions"].append({
                "frame": frame_id,
                "x": center_x,
                "y": center_y,
                "box": [x1, y1, x2, y2]
            })

    frame_id += 1

cap.release()

trajectories_json = {
    str(track_id): data
    for track_id, data in trajectories.items()
}

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(trajectories_json, file, indent=4)

print(f"Frames procesados: {frame_id}")
print(f"Objetos encontrados: {len(trajectories)}")
print(f"Recortes guardados en: {crops_dir}")
print(f"Trayectorias guardadas en: {output_path}")

In [ ]:
# Visualización del tracking YOLO con control de reproducción y selección de objetos
# Depende de la celda anterior, que genera el archivo "trajectories.json" y los recortes de objetos en la carpeta "object_crops"

import cv2
import json
import os
import numpy as np

video_path = "images/calle.mp4"
trajectories_path = "trajectories.json"
window_name = "Trayectoria del objeto"

with open(trajectories_path, "r", encoding="utf-8") as file:
    trajectories = json.load(file)

object_ids = sorted(trajectories.keys(), key=int)

if not object_ids:
    raise ValueError("No se han encontrado objetos en trajectories.json")

selected_index = 0
frame_id = 0
playing = False
scroll_offset = 0

cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ICON_WIDTH = 100
ICON_HEIGHT = 85
PANEL_HEIGHT = 110

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
visible_count = max(1, frame_width // ICON_WIDTH)
max_scroll = max(0, len(object_ids) - visible_count)


def slider_callback(value):
    global scroll_offset
    scroll_offset = min(value, max_scroll)


def change_object(index):
    global selected_index, frame_id

    selected_index = index % len(object_ids)
    frame_id = 0


def mouse_callback(event, x, y, flags, param):
    # Solo gestionar clics sobre los iconos.
    # No usamos la rueda del ratón.
    if event == cv2.EVENT_LBUTTONDOWN and y < PANEL_HEIGHT:
        clicked_index = (x // ICON_WIDTH) + scroll_offset

        if 0 <= clicked_index < len(object_ids):
            change_object(clicked_index)


def create_icon(object_id, selected):
    crop_path = trajectories[object_id]["crop_path"]
    icon = cv2.imread(crop_path)

    if icon is None:
        icon = np.zeros((70, 90, 3), dtype="uint8")
    else:
        scale = min(
            80 / icon.shape[1],
            65 / icon.shape[0]
        )

        icon = cv2.resize(
            icon,
            (
                max(1, int(icon.shape[1] * scale)),
                max(1, int(icon.shape[0] * scale))
            )
        )

    canvas = np.zeros((ICON_HEIGHT, ICON_WIDTH, 3), dtype="uint8")

    y_offset = (ICON_HEIGHT - icon.shape[0]) // 2
    x_offset = (ICON_WIDTH - icon.shape[1]) // 2

    canvas[
        y_offset:y_offset + icon.shape[0],
        x_offset:x_offset + icon.shape[1]
    ] = icon

    color = (0, 255, 0) if selected else (150, 150, 150)

    cv2.rectangle(
        canvas,
        (0, 0),
        (ICON_WIDTH - 1, ICON_HEIGHT - 1),
        color,
        3
    )

    cv2.putText(
        canvas,
        f"ID {object_id}",
        (8, ICON_HEIGHT - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1
    )

    return canvas


cv2.namedWindow(window_name)
cv2.setMouseCallback(window_name, mouse_callback)

# Slider para desplazar la lista de iconos
cv2.createTrackbar(
    "Objetos",
    window_name,
    0,
    max(1, max_scroll),
    slider_callback
)

try:
    while True:
        selected_id = object_ids[selected_index]
        selected_object = trajectories[selected_id]
        positions = selected_object["positions"]

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()

        if not ret:
            break

        # Dibujar la trayectoria completa
        points = [
            (int(position["x"]), int(position["y"]))
            for position in positions
        ]

        for point_a, point_b in zip(points, points[1:]):
            cv2.line(
                frame,
                point_a,
                point_b,
                (0, 255, 255),
                3
            )

        # Buscar la posición del objeto en el frame actual
        current_position = next(
            (
                position
                for position in positions
                if position["frame"] == frame_id
            ),
            None
        )

        if current_position is not None:
            current_point = (
                int(current_position["x"]),
                int(current_position["y"])
            )

            # Punto verde del frame actual
            cv2.circle(
                frame,
                current_point,
                8,
                (0, 255, 0),
                -1
            )

            x1, y1, x2, y2 = current_position["box"]

            cv2.rectangle(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (0, 255, 0),
                2
            )

        cv2.putText(
            frame,
            f"ID: {selected_id} | "
            f"Clase: {selected_object['class_name']}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Frame: {frame_id + 1}/{total_frames}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        # Crear panel superior de iconos
        panel = np.zeros(
            (PANEL_HEIGHT, frame.shape[1], 3),
            dtype="uint8"
        )

        visible_ids = object_ids[
            scroll_offset:scroll_offset + frame.shape[1] // ICON_WIDTH
        ]

        for position, object_id in enumerate(visible_ids):
            object_index = scroll_offset + position

            icon = create_icon(
                object_id,
                object_index == selected_index
            )

            x_offset = position * ICON_WIDTH

            panel[
                10:10 + ICON_HEIGHT,
                x_offset:x_offset + ICON_WIDTH
            ] = icon

        cv2.putText(
            panel,
            "Rueda: desplazar | A/D o clic: cambiar objeto",
            (10, PANEL_HEIGHT - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (200, 200, 200),
            1
        )

        display = cv2.vconcat((panel, frame))
        cv2.imshow(window_name, display)

        key = cv2.waitKeyEx(30) & 0xFF

        # ESC o Q
        if key in (27, ord("q")):
            break

        # ESPACIO: reproducir/pausar
        elif key == ord(" "):
            playing = not playing

        # A: objeto anterior
        elif key == ord("a"):
            change_object(selected_index - 1)

            if selected_index < scroll_offset:
                scroll_offset = selected_index
                cv2.setTrackbarPos(
                    "Objetos",
                    window_name,
                    scroll_offset
                )

        # D: objeto siguiente
        elif key == ord("d"):
            change_object(selected_index + 1)

            if selected_index >= scroll_offset + visible_count:
                scroll_offset = selected_index - visible_count + 1
                scroll_offset = min(scroll_offset, max_scroll)

                cv2.setTrackbarPos(
                    "Objetos",
                    window_name,
                    scroll_offset
                )

        if playing:
            frame_id += 1

            if frame_id >= total_frames:
                frame_id = total_frames - 1
                playing = False

finally:
    cap.release()
    cv2.destroyAllWindows()

print("Visualizador cerrado.")

#### Guardamos los resultados de detección en el frame número 90 para utilizarlos en la siguiente práctica

In [ ]:
# Guardado de los resultados del frame 90 para trabajar posteriormente con SAM

import cv2
import json
import os

video_path = "images/calle.mp4"
frame_number = 90
frame_path = "images/calle_frame_90.jpg"
annotated_frame_path = "images/calle_frame_90_annotated.jpg"
results_path = "calle_frame_90_results.json"

if "model_yolo" not in globals():
    from ultralytics import YOLO
    model_yolo = YOLO("yolo26n.pt")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    cap.release()
    raise RuntimeError(f"No se pudo abrir el vídeo: {video_path}")

cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
ret, frame_90 = cap.read()
cap.release()

if not ret or frame_90 is None:
    raise RuntimeError(f"No se pudo leer el frame {frame_number}")

# Detectar personas en un frame aislado. Para una imagen no usamos tracking:
# el tracker puede descartar detecciones al no tener contexto temporal.
results_90 = model_yolo.track(
    frame_90,
    classes=[0],
    imgsz=1920,
    conf=0.01,
    verbose=False,
    persist=True
)[0]

boxes_90 = results_90.boxes
frame_90_boxes = []

for index, box in enumerate(boxes_90.xyxy.cpu().tolist()):
    class_id = int(boxes_90.cls[index].item())
    confidence = float(boxes_90.conf[index].item())
    track_id = None

    x1, y1, x2, y2 = map(int, box)
    frame_90_boxes.append({
        "index": index,
        "track_id": track_id,
        "class_id": class_id,
        "class_name": model_yolo.names[class_id],
        "confidence": confidence,
        "xyxy": [x1, y1, x2, y2],
        "center": [(x1 + x2) // 2, (y1 + y2) // 2]
    })

frame_90_metadata = {
    "video_path": video_path,
    "frame_number": frame_number,
    "image_path": frame_path,
    "annotated_image_path": annotated_frame_path,
    "image_width": int(frame_90.shape[1]),
    "image_height": int(frame_90.shape[0]),
    "boxes": frame_90_boxes
}

os.makedirs(os.path.dirname(frame_path), exist_ok=True)
cv2.imwrite(frame_path, frame_90)
cv2.imwrite(annotated_frame_path, results_90.plot())

with open(results_path, "w", encoding="utf-8") as file:
    json.dump(frame_90_metadata, file, indent=4)

# Variables reutilizables en las siguientes celdas, por ejemplo para SAM.
frame_90_rgb = cv2.cvtColor(frame_90, cv2.COLOR_BGR2RGB)
frame_90_boxes_xyxy = [item["xyxy"] for item in frame_90_boxes]

print(f"Frame guardado en: {frame_path}")
print(f"Resultados guardados en: {results_path}")
print(f"Personas detectadas: {len(frame_90_boxes)}")